# Notebook 02: Validação e Exploração da Camada Silver
**Projeto Integrador: Da Ingestão à Decisão (ENEM + ANEEL)**

Este notebook tem como finalidade inspecionar e auditar a **Camada Silver** do projeto:
1. Verificação dos esquemas e tipagem forte da ANEEL e do ENEM;
2. Conformidade rigorosa com a LGPD (expurgo de PII);
3. Validação dos Contratos de Dados (Data Contracts) e Quarentena;
4. Auditoria de integridade do cruzamento relacional (Join 100%);
5. Estatísticas descritivas das métricas que alimentarão a Camada Gold.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Localização do diretório de dados da Camada Silver
base_dir = Path.cwd()
silver_dir = base_dir / 'data' / 'silver'
if not silver_dir.exists():
    silver_dir = base_dir.parent / 'data' / 'silver'

print(f'Diretório Silver localizado em: {silver_dir.resolve()}')

## 1. Inspeção da Camada Silver da ANEEL (API REST JSON)
Os indicadores de continuidade DEC (duração em horas) e FEC (frequência de quedas) foram agregados a nível municipal por competência (ano/mês) a partir da resolução N:M com a tabela `indqual-municipio`.

In [ ]:
df_aneel = pd.read_parquet(silver_dir / 'aneel' / 'aneel_silver.parquet')
print(f'Total de registros mensais na Silver ANEEL: {len(df_aneel):,}')
print(f'Total de municípios únicos cobertos: {df_aneel["codigo_municipio"].nunique()}')
print(f'Anos cobertos: {sorted(df_aneel["ano"].unique().tolist())}')
df_aneel.head(10)

## 2. Inspeção da Camada Silver do ENEM (Microdados Agregados)
Conformidade com a LGPD: o identificador individual `NU_INSCRICAO` foi expurgado, e os dados individuais foram agregados por `codigo_municipio` e `ano`.

In [ ]:
df_enem = pd.read_parquet(silver_dir / 'enem' / 'enem_silver.parquet')
print(f'Total de observações município-ano no ENEM: {len(df_enem):,}')
print(f'Municípios únicos cobertos: {df_enem["codigo_municipio"].nunique()}')
assert 'NU_INSCRICAO' not in df_enem.columns, 'ERRO: NU_INSCRICAO não pode existir na Silver!'
print('Conformidade LGPD: [OK] Nenhum dado pessoal identificável presente.')
df_enem.head(10)

## 3. Auditoria do JOIN Relacional (Silver Joined)
Cruzamento pelo código IBGE do município e ano, gerando a base integrada com todas as competências mensais.

In [ ]:
df_joined = pd.read_parquet(silver_dir / 'joined' / 'silver_joined.parquet')
print(f'Total de registros integrados: {len(df_joined):,}')
print(f'Municípios casados no Join: {df_joined["codigo_municipio"].nunique()} de 144 (100% de cobertura)')
df_joined[['codigo_municipio', 'NomMunicipio', 'ano', 'mes', 'dec_horas_mensal', 'fec_freq_mensal', 'total_inscritos', 'taxa_abstencao']].head(10)

## 4. Estatísticas Descritivas e Perfilamento das Variáveis

In [ ]:
metricas = ['dec_horas_mensal', 'dec_horas_max', 'fec_freq_mensal', 'fec_freq_max', 'total_inscritos', 'taxa_abstencao']
df_joined[metricas].describe().round(2)